# Profiling 实操：基线 vs 融合对比

上一节我们理解了融合算子的实现。本节运行两组 profiling 训练——**不启用融合算子**（baseline）和**启用融合算子**（fused），然后用 `analyze_profiling.py` 自动对比。

---

## Profiling 配置说明

在 `config_registry.py` 中，profiling 相关配置集中在 `ProfilingConfig`：

```python
profiling=ProfilingConfig(
    enable_profiling=False,      # ← 默认关闭，profiling 时改为 True
    enable_online_parse=True,    # 训练结束后自动解析 trace
    profile_ranks=[0],           # profile rank 0
    profile_step_start=5,        # 从第 5 步开始采集
    profile_step_end=6,          # 到第 6 步结束（采集 1 步）
    profile_record_shapes=True,  # 记录张量形状
    profile_with_memory=True,    # 记录显存使用
)
```

> **`profile_step_start=5`**：跳过前 5 步（含初始化、第一次 CUDA kernel 编译等），从第 6 步采集——此时 kernel 已预热，数据更能反映稳态性能。

### Profiling 的 batch_size 设置

第 3 章训练用 `global_batch_size=64` 追求收敛质量。Profiling 时可选择改为 `gbs=4`：

```text
gbs=64 → 32 次前向，梯度累计一次 → 追求收敛质量，慢但模型收敛好
gbs=4  →  2 次前向，梯度累计一次 → 追求跑得快，只看算子耗时
```

见以下训练配置中的  --training.global_batch_size=4 。

> Profiling 只关心算子的**单步耗时分布**，不关心收敛效果。用小 batch_size 减少等待时间，快速出 profile 结果。

---

## 训练命令

### baseline 配置

在 `config_registry.py` 中，使用不含 `model_converters` 的配置：

```python
def _qwen3_1_7b_converters() -> ModelConvertersContainer.Config:
    return ModelConvertersContainer.Config(
        converters=[
          # 基线 - 不启用任何 converters
        ],
    )

```

启动训练 + profiling：

```bash
NGPU=2 \
MODULE=torchtitan_npu.models.qwen3 \
CONFIG=sft_qwen3_1_7b_wordle \
bash scripts/run_train.sh \
  --training.steps=10 \
  --training.global_batch_size=4 \
  --profiling.enable_profiling
  --dataloader.dataset_path ./assets/data/wordle
```

### fused 配置

同一份 `config_registry.py`，启用 `model_converters`：

```python
def _qwen3_1_7b_converters() -> ModelConvertersContainer.Config:
    return ModelConvertersContainer.Config(
        converters=[
          # 启用 npu_rms_norm
          get_model_converter_config("npu_rms_norm"),
        ],
    )

```

```bash
NGPU=2 \
MODULE=torchtitan_npu.models.qwen3 \
CONFIG=sft_qwen3_1_7b_wordle_fused \
bash scripts/run_train.sh \
  --training.steps=10 \
  --training.global_batch_size=4 \
  --profiling.enable_profiling
  --dataloader.dataset_path ./assets/data/wordle
```

> 如果之前跑过训练，先清理旧 checkpoint：`rm -rf outputs/checkpoint_wordle_sft/`

### 关键差异

两次运行唯一的不同是 `model_converters`：

| 配置 | model_converters | 效果 |
|------|-----------------|------|
| baseline | `[]`（空） | 原生 PyTorch RMSNorm，6 kernel/次 |
| fused | `["npu_rms_norm"]` | NPU 融合 RMSNorm，1 kernel/次 |

两次运行后，profiling 输出分别落在 `outputs/profile_traces/` 下的两个子目录中（目录名带时间戳）。

---

## 运行 analyze_profiling.py

训练完成后，将 `tools/analyze_profile.py` 移动到 `torchtitan-npu` 的 `scripts` 目录，并用以下命令自动发现并对比两组 trace：

```bash
python scripts/analyze_profile.py --profile_dir ./outputs/profile_traces
```

脚本会自动：

1. **发现 trace**：扫描 `profile_traces/` 下的所有子目录，找到 `ASCEND_PROFILER_OUTPUT/trace_view.json`。
2. **排序**：按文件修改时间排序——先跑的 = baseline，后跑的 = fused。
3. **解析**：加载 JSON，按算子名分类，统计 ACL kernel 调用次数和耗时。
4. **对比**：输出差异表格，验证融合算子是否生效。

### 预期输出

```text
═══════════════════════════════════════════════════════
  Ascend NPU 融合算子 Profiling 分析工具
═══════════════════════════════════════════════════════
🔍 扫描 profiling 输出目录: ./outputs/profile_traces
  📋 发现 trace: profile_baseline_xxx (时间戳: ...)
  📋 发现 trace: profile_fused_xxx (时间戳: ...)

  ✅ 自动排序结果（按时间）:
    1. baseline: profile_baseline_xxx
    2. fused:    profile_fused_xxx

✅ 分析完成。
```

下一节，我们将深入解读这些对比输出的含义。

## 练习

1. （判断题）Profiling 时 `profile_step_start=5` 是为了跳过前 5 步的 kernel 首次编译、显存预热等一次性开销，采集稳态性能数据。

2. （判断题）Profiling 时把 `global_batch_size` 从 64 改成 4，目的是减少单步训练时间以提高收敛质量。

3. （单选题）`analyze_profiling.py` 如何自动区分 baseline 和 fused trace？
    A. 读取 trace 文件名中的 "baseline"/"fused" 关键字
    B. 按 `trace_view.json` 的文件修改时间排序——先跑的 = baseline，后跑的 = fused
    C. 比较两个 trace 的 loss 值
    D. 随机指定

In [ ]:
!cat ./answer/04.03_answer.txt